# Notebook 1 — Téléchargement et préparation des données

**Projet :** artificialisation du plateau de Saclay entre 2017 et 2024 (UNet, Sentinel-2).

Ce notebook produit, dans le dossier `data_saclay/` :

| Fichier | Contenu |
|---|---|
| `s2_2017.tif`, `s2_2021.tif`, `s2_2024.tif` | Images Sentinel-2 L2A, 4 bandes (B02 bleu, B03 vert, B04 rouge, B08 proche infrarouge), 10 m |
| `worldcover_2021_brut.tif` | Carte ESA WorldCover 2021 (11 classes d'origine) |
| `labels_2021.tif` | Labels regroupés en 5 classes, alignés pixel à pixel sur les images |
| `metadonnees.json` | Dates retenues, identifiants des images, taux de nuages (utile pour le rapport) |

**Principe :** pour chaque année, on cherche les images d'été peu nuageuses, on mesure la part de nuages **sur notre zone** grâce à la bande SCL (classification de scène fournie avec Sentinel-2), et on garde la meilleure date.

Toutes les données viennent du **Microsoft Planetary Computer** : aucun compte n'est nécessaire.

In [ ]:
# Installation (une seule fois). Redémarre le kernel après si VS Code le demande.
%pip install pystac-client planetary-computer odc-stac rioxarray dask

## 1. Configuration

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import xarray as xr
import rioxarray  # active l'accesseur .rio sur les DataArray
import rasterio
import pystac_client
import planetary_computer
from odc.stac import load as stac_load

DATA_DIR = Path("data_saclay")
DATA_DIR.mkdir(exist_ok=True)

# Zone d'étude : plateau de Saclay et alentours (lon_min, lat_min, lon_max, lat_max), environ 40 x 37 km
BBOX = [1.92, 48.55, 2.46, 48.88]
CRS = "EPSG:32631"   # UTM zone 31N, projection métrique adaptée à l'Île-de-France
RES = 10             # résolution en mètres (native pour B02, B03, B04, B08)

S2_BANDS = ["B02", "B03", "B04", "B08"]

# Fenêtres de recherche : l'été, pour avoir une végétation comparable d'une année à l'autre
PERIODES = {
    "2017": "2017-06-01/2017-09-30",
    "2021": "2021-06-01/2021-09-30",
    "2024": "2024-06-01/2024-09-30",
}

MAX_NUAGES_TUILE = 20    # % de nuages max annoncé pour la tuile entière (pré-filtre)
MAX_INVALIDE_ZONE = 0.01 # part max de pixels nuageux ou vides sur NOTRE zone (1 %)
N_CANDIDATS = 8          # nombre de dates testées au maximum par année

## 2. Connexion au catalogue

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,  # signe automatiquement les liens de téléchargement
)
print("Catalogue connecté")

## 3. Choix de la meilleure date pour chaque année

La zone peut être à cheval sur plusieurs tuiles Sentinel-2 : on regroupe donc les images **par jour** et on les assemble (`groupby="solar_day"`).

Pour mesurer les nuages sur la zone, on charge la bande **SCL** en basse résolution (60 m, rapide) et on compte les pixels des classes : 0 (vide), 1 (saturé), 3 (ombre de nuage), 8 et 9 (nuages), 10 (cirrus).

In [ ]:
CLASSES_SCL_INVALIDES = [0, 1, 3, 8, 9, 10]

def chercher_items_par_jour(periode):
    recherche = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=BBOX,
        datetime=periode,
        query={"eo:cloud_cover": {"lt": MAX_NUAGES_TUILE}},
    )
    par_jour = {}
    for item in recherche.item_collection():
        jour = item.datetime.date().isoformat()
        par_jour.setdefault(jour, []).append(item)
    return par_jour


def part_invalide(items):
    scl = stac_load(items, bands=["SCL"], crs=CRS, resolution=60,
                    bbox=BBOX, groupby="solar_day")["SCL"].isel(time=0).values
    return float(np.isin(scl, CLASSES_SCL_INVALIDES).mean())


def choisir_meilleure_date(annee):
    par_jour = chercher_items_par_jour(PERIODES[annee])
    if not par_jour:
        raise RuntimeError(f"Aucune image trouvée pour {annee} : élargis la période ou MAX_NUAGES_TUILE.")
    print(f"{len(par_jour)} dates candidates")

    # On teste d'abord les jours dont les tuiles sont les moins nuageuses
    jours = sorted(par_jour, key=lambda j: np.mean([it.properties["eo:cloud_cover"] for it in par_jour[j]]))
    meilleur_jour, meilleure_part = None, 1.0
    for jour in jours[:N_CANDIDATS]:
        part = part_invalide(par_jour[jour])
        print(f"  {jour} : {part:.2%} de pixels nuageux ou vides sur la zone")
        if part < meilleure_part:
            meilleur_jour, meilleure_part = jour, part
        if part <= MAX_INVALIDE_ZONE:
            break
    return meilleur_jour, par_jour[meilleur_jour], meilleure_part

## 4. Téléchargement des images Sentinel-2

**Point important pour le rapport :** depuis le 25 janvier 2022, l'ESA ajoute un décalage de **+1000** aux valeurs Sentinel-2 L2A (traitement « baseline 04.00 »). Sans correction, l'image de 2024 paraîtrait plus claire que celles de 2017 et 2021, et le modèle se tromperait. On retire donc ce décalage quand la baseline est ≥ 4.0, pour que les trois années soient comparables.

In [ ]:
def charger_s2(items):
    ds = stac_load(items, bands=S2_BANDS, crs=CRS, resolution=RES, bbox=BBOX,
                   groupby="solar_day", chunks={"x": 2048, "y": 2048})
    da = ds[S2_BANDS].to_array("band").isel(time=0).compute()

    baseline = float(items[0].properties.get("s2:processing_baseline", "0") or 0)
    if baseline >= 4.0:
        # Harmonisation : on retire le décalage de 1000, en gardant 0 comme valeur « vide »
        vide = da == 0
        da = (da.astype("int32") - 1000).clip(min=1).where(~vide, 0)
    return da.astype("uint16"), baseline


def enregistrer(da, chemin, nodata):
    da = da.assign_coords(band=np.arange(1, da.sizes["band"] + 1))
    da = da.rio.write_crs(CRS).rio.write_nodata(nodata)
    da.rio.to_raster(chemin, compress="deflate")


metadonnees = {"bbox": BBOX, "crs": CRS, "resolution_m": RES, "sentinel2": {}}

for annee in PERIODES:
    print(f"\n===== {annee} =====")
    jour, items, part = choisir_meilleure_date(annee)
    print(f"-> Date retenue : {jour} ({part:.2%} de pixels invalides). Téléchargement...")

    da, baseline = charger_s2(items)
    chemin = DATA_DIR / f"s2_{annee}.tif"
    enregistrer(da, chemin, nodata=0)
    print(f"   Enregistré : {chemin} | taille {da.sizes['x']} x {da.sizes['y']} | baseline {baseline}")

    metadonnees["sentinel2"][annee] = {
        "date": jour,
        "part_invalide": part,
        "processing_baseline": baseline,
        "items": [it.id for it in items],
    }

## 5. Labels : ESA WorldCover 2021

WorldCover a 11 classes. On les regroupe en **5 classes** adaptées à l'Île-de-France et à la question de l'artificialisation :

| Nouvelle classe | Classes WorldCover d'origine |
|---|---|
| 0 — Bâti | 50 (built-up) |
| 1 — Forêt / arbres | 10 (tree cover), 95 (mangroves) |
| 2 — Cultures | 40 (cropland) |
| 3 — Prairies / autre | 20, 30, 60, 70, 90, 100 (arbustes, prairies, sol nu, neige, zones humides, mousses) |
| 4 — Eau | 80 (permanent water) |

La valeur **255** signifie « pas de donnée » : elle sera ignorée pendant l'entraînement.

In [ ]:
wc_items = catalog.search(
    collections=["esa-worldcover"], bbox=BBOX, datetime="2021-01-01/2021-12-31"
).item_collection()
print(f"{len(wc_items)} tuile(s) WorldCover 2021 trouvée(s)")

wc = stac_load(wc_items, bands=["map"], crs=CRS, resolution=RES, bbox=BBOX,
               groupby="solar_day", resampling="nearest",
               chunks={"x": 2048, "y": 2048})["map"].isel(time=0).compute()

enregistrer(wc.expand_dims("band"), DATA_DIR / "worldcover_2021_brut.tif", nodata=0)

NOMS_CLASSES = ["Bâti", "Forêt / arbres", "Cultures", "Prairies / autre", "Eau"]
CORRESPONDANCE = {50: 0, 10: 1, 95: 1, 40: 2, 20: 3, 30: 3, 60: 3, 70: 3, 90: 3, 100: 3, 80: 4}

table = np.full(256, 255, dtype=np.uint8)  # tout ce qui n'est pas listé devient 255 (pas de donnée)
for code_wc, nouvelle in CORRESPONDANCE.items():
    table[code_wc] = nouvelle

labels = wc.copy(data=table[wc.values.astype(np.uint8)])
enregistrer(labels.expand_dims("band"), DATA_DIR / "labels_2021.tif", nodata=255)
print("Labels enregistrés :", DATA_DIR / "labels_2021.tif")

metadonnees["worldcover"] = {
    "items": [it.id for it in wc_items],
    "classes": NOMS_CLASSES,
    "correspondance": {str(k): v for k, v in CORRESPONDANCE.items()},
}

## 6. Vérifications

In [ ]:
# Toutes les couches doivent avoir exactement la même grille (taille, projection, géoréférencement)
chemins = [DATA_DIR / f"s2_{a}.tif" for a in PERIODES] + [DATA_DIR / "labels_2021.tif"]
reference = None
for chemin in chemins:
    with rasterio.open(chemin) as src:
        grille = (src.width, src.height, src.crs.to_string(), tuple(round(v, 3) for v in src.transform[:6]))
        print(f"{chemin.name:22s} {src.count} bande(s) | {src.width} x {src.height} | {src.crs} | {src.dtypes[0]}")
        if reference is None:
            reference = grille
        elif grille != reference:
            print("   /!\\ Grille différente de la première couche : à signaler !")

# Répartition des classes
valeurs, effectifs = np.unique(labels.values, return_counts=True)
total = effectifs[valeurs != 255].sum()
print("\nRépartition des classes (2021) :")
for v, n in zip(valeurs, effectifs):
    nom = NOMS_CLASSES[v] if v < len(NOMS_CLASSES) else "Pas de donnée"
    part = f"{n / total:6.1%}" if v != 255 else "  ignoré"
    print(f"  {v:3d} {nom:18s} {part}")

In [ ]:
def rgb(chemin, bornes=None):
    with rasterio.open(chemin) as src:
        img = src.read([3, 2, 1]).astype("float32")  # B04, B03, B02 -> rouge, vert, bleu
    img = np.moveaxis(img, 0, -1)
    if bornes is None:
        bornes = np.percentile(img[img > 0], [2, 98])
    return np.clip((img - bornes[0]) / (bornes[1] - bornes[0]), 0, 1), bornes

# Même étirement de contraste pour les trois années : si une image paraît beaucoup plus claire
# ou plus sombre que les autres, c'est un problème de radiométrie à corriger avant d'aller plus loin.
_, bornes = rgb(DATA_DIR / "s2_2021.tif")

couleurs = ListedColormap(["#d7191c", "#1a9641", "#fdae61", "#a6d96a", "#2b83ba"])
fig, axes = plt.subplots(1, 4, figsize=(24, 6))
for ax, annee in zip(axes[:3], PERIODES):
    img, _ = rgb(DATA_DIR / f"s2_{annee}.tif", bornes)
    ax.imshow(img)
    ax.set_title(f"Sentinel-2 {annee} ({metadonnees['sentinel2'][annee]['date']})")
    ax.axis("off")

lab = np.ma.masked_equal(labels.values, 255)
axes[3].imshow(lab, cmap=couleurs, vmin=0, vmax=4, interpolation="nearest")
axes[3].set_title("Labels WorldCover 2021 (5 classes)")
axes[3].axis("off")
for i, nom in enumerate(NOMS_CLASSES):
    axes[3].plot([], [], "s", color=couleurs(i), label=nom, markersize=10)
axes[3].legend(loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig(DATA_DIR / "apercu_donnees.png", dpi=120)
plt.show()

In [ ]:
with open(DATA_DIR / "metadonnees.json", "w", encoding="utf-8") as f:
    json.dump(metadonnees, f, indent=2, ensure_ascii=False)
print(json.dumps(metadonnees["sentinel2"], indent=2))

## Ce qu'il faut vérifier avant de passer au notebook 2

1. **Les trois images sont nettes**, sans nuages visibles, et ont des couleurs comparables.
2. **Les quatre couches ont la même grille** : aucun message « /!\ Grille différente ».
3. **La répartition des classes est plausible** : beaucoup de bâti au nord-est (banlieue sud de Paris), des cultures sur le plateau, de la forêt dans les vallées.

Garde `metadonnees.json` et `apercu_donnees.png` : ils serviront directement pour la partie *Méthodes et données* du rapport.